In [12]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
# ██████╗  █████╗ ██████╗  █████╗ ███╗   ███╗███████╗
# ██╔══██╗██╔══██╗██╔══██╗██╔══██╗████╗ ████║██╔════╝
# ██████╔╝███████║██████╔╝███████║██╔████╔██║███████╗
# ██╔═══╝ ██╔══██║██╔══██╗██╔══██║██║╚██╔╝██║╚════██║
# ██║     ██║  ██║██║  ██║██║  ██║██║ ╚═╝ ██║███████║
# ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═╝╚═╝     ╚═╝╚══════╝
#
#  Smart APS V6  —  VT-Only Indent-Based Planning
#
#  Planning logic:
#    daily_indent  = monthly_indent / working_days
#    today_target  = max(0, daily_indent − inventory)
#    working_days  = calendar days in month − number of Sundays
#
#  Skip rules (part excluded from planning entirely):
#    • monthly_indent < 100
#    • monthly_indent / rate  ≤ 2 hours  (whole indent is trivial)
#
#  Zero-inventory rule:
#    If inventory = 0, part is FORCED into plan regardless of
#    skip rules. Parts sorted by highest daily_indent first
#    (greedy fill) so most critical parts always get time.
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS  (change these every morning)
# -------------------------------------------------------------
# Only two lines to update each day:
#   PLANNING_DATE  : today
#   INDENT_MONTH   : only change when the month rolls over
# =============================================================

PLANNING_DATE = date(2026, 3, 19)   # ← change daily
INDENT_MONTH  = date(2026, 3,  1)   # ← change when month rolls over

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS    = 22        # machine hours available per shift
MIN_RUN_HOURS      = 4         # minimum run block per part
TARGET_DAYS_INV    = 3         # ideal inventory buffer (days)
MACHINE_STATE_FILE = "machine_state.json"

# Skip thresholds
MIN_DAILY_INDENT   = 150       # skip if daily_indent ≤ this (absolute, no exceptions)
MIN_INDENT_HOURS   = 4.0       # skip if whole monthly indent takes ≤ this many hours
                               # (monthly_indent / rate ≤ MIN_INDENT_HOURS)
                               # Both rules are absolute — zero inventory is no exception

MAX_OVERPRODUCTION_DAYS = 5    # max total stock (inv + produced) = this × daily_indent
                               # prevents any part from consuming machine time beyond
                               # 5 days of supply while other parts are still unmet

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_Plan_{PLANNING_DATE.strftime('%Y%m%d')}Indent-v2.xlsx"

# =============================================================
# SECTION 4 — WORKING DAYS CALCULATION
# -------------------------------------------------------------
# working_days = calendar days in INDENT_MONTH − Sundays
# Example: March 2026 → 31 days, 5 Sundays → 26 working days
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*62}")
print(f"  Smart APS V6  —  VT-Only Indent-Based Planning")
print(f"  Planning date  : {PLANNING_DATE}")
print(f"  Indent month   : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Total days     : {TOTAL_DAYS}")
print(f"  Sundays        : {SUNDAY_COUNT}")
print(f"  Working days   : {WORKING_DAYS}  (used as divisor)")
print(f"  Daily target   : monthly_indent / {WORKING_DAYS}")
print(f"  Today target   : max(0, daily_indent - inventory)")
print(f"  Skip rule 1    : daily_indent ≤ {MIN_DAILY_INDENT} qty  (absolute, no exceptions)")
print(f"  Skip rule 2    : monthly_indent / rate  ≤ {MIN_INDENT_HOURS}h  (absolute, no exceptions)")
print(f"  Zero-inv rule  : NONE — skip rules apply even when inventory = 0")
print(f"  Over-prod cap  : {MAX_OVERPRODUCTION_DAYS} × daily_indent  (inv + produced never exceeds this)")
print(f"  Changeover     : none on first run (clean slate) — charged from day 2 onwards")
print(f"{'='*62}\n")

# =============================================================
# SECTION 5 — LOAD DATA  (VT sheet is the single source of truth)
# =============================================================

print("Loading data...")
vt_parts_raw = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw    = pd.read_excel(changeover_path, sheet_name="VT_Changeover")

# =============================================================
# SECTION 6 — READ ALL DATA FROM VT SHEET
# -------------------------------------------------------------
# VT sheet is the single source of truth. It must contain:
#   • Part          — part code
#   • Cycle time    — cycle time in seconds
#   • Cavity        — number of cavities
#   • Inventory     — current stock
#   • Indent        — monthly indent quantity
#   • Category      — Runner / Repeater / Stranger
#
# Rate (units/hour) = (3600 / cycle_time_seconds) × cavity_count
#
# All columns are matched case-insensitively.
# A clear error is raised if any required column is missing.
# =============================================================

def find_col(df, name, sheet):
    """Case-insensitive column finder. Raises clear error if missing."""
    match = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available columns: {list(df.columns)}"
        )
    return match

# Locate every required column in VT sheet
vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")

print(f"  VT sheet columns found:")
print(f"    Part       → '{vt_col_part}'")
print(f"    Cycle time → '{vt_col_cycletime}'")
print(f"    Cavity     → '{vt_col_cavity}'")
print(f"    Inventory  → '{vt_col_inventory}'")
print(f"    Indent     → '{vt_col_indent}'")

# Drop blank part rows, deduplicate
data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

# Rate formula:
#   cycle_time = time per piece per cavity (seconds)
#   For N cavities: shot takes cycle_time × N seconds, produces N pieces
#   → Rate = (3600 / (cycle_time × cavity)) × cavity = 3600 / cycle_time
#   Cavities cancel — rate depends only on cycle_time.
#   Example: cycle_time=15s, cavity=2 → shot=30s, 2 pcs → 240 pcs/hr = 3600/15
data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["_cv"] = pd.to_numeric(data[vt_col_cavity],    errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]   # cavities cancel — see formula above

# Parts with valid rate (cycle time present and non-zero)
# Cavity is retained for reference but does NOT affect rate.
data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()   # only zero/missing cycle time

print(f"\n  VT parts in sheet            : {len(data)}")
print(f"  Parts with valid rate        : {len(data_valid)}")
if len(data_zero_rate) > 0:
    print(f"  Parts with zero/missing cycle time : {len(data_zero_rate)}")
    for _, r in data_zero_rate.iterrows():
        ct = r[vt_col_cycletime]
        print(f"    {str(r['Material']):30s}  cycle_time={ct}  (cavity={r[vt_col_cavity]} — not used in rate)")

# =============================================================
# SECTION 7 — BUILD LOOKUP DICTIONARIES  (all from VT sheet)
# =============================================================

def safe_dict_from_df(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

# Use data_valid for rate (only parts with computable rate)
# Use full data for inventory and indent (available for all parts)
inventory = safe_dict_from_df(data,       "Material", vt_col_inventory)
rate      = safe_dict_from_df(data_valid, "Material", "Rate")

# =============================================================
# SECTION 7A — INDENT DICTIONARIES  (read from VT sheet)
# -------------------------------------------------------------
# indent_monthly   : {part → monthly indent qty}
# indent_daily     : {part → monthly / working_days}
# today_target_qty : {part → max(0, daily_indent − inventory)}
# =============================================================

print("\nReading indent data from VT sheet...")
indent_monthly = safe_dict_from_df(data, "Material", vt_col_indent)

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# ── Skip flags ───────────────────────────────────────────────
# A part is skipped ONLY when BOTH conditions below are true:
#   condition met  AND  inventory > 0
# If inventory == 0, part is NEVER skipped — it must be made.

def should_skip(part):
    """
    Returns (skip: bool, reason: str).
    Rules are absolute — zero inventory is NO exception.
    Skip if:  daily_indent ≤ MIN_DAILY_INDENT (150)
           OR monthly_indent / rate ≤ MIN_INDENT_HOURS (4h)
    """
    monthly = indent_monthly.get(part, 0.0)
    daily   = indent_daily.get(part, 0.0)
    r       = rate.get(part, 1.0)

    # Rule 1: daily indent too small — not worth planning
    if daily <= MIN_DAILY_INDENT:
        return True, (f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} qty threshold")

    # Rule 2: entire monthly indent is trivial to produce
    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, (f"Whole monthly indent takes only {indent_hrs:.2f}h "
                      f"≤ {MIN_INDENT_HOURS}h threshold — not worth planning")

    return False, ""

# Print skip summary
skipped_parts = {p: should_skip(p) for p in indent_monthly}
n_skip = sum(1 for skip, _ in skipped_parts.values() if skip)
print(f"\n  Indent summary:")
print(f"    Total VT parts with indent data : {len(indent_monthly)}")
print(f"    Parts skipped (low/trivial)     : {n_skip}  ← never planned, no exceptions")
print(f"    Skip rule 1 : daily_indent ≤ {MIN_DAILY_INDENT} qty  (absolute — zero inv is no exception)")
print(f"    Skip rule 2 : monthly_indent / rate ≤ {MIN_INDENT_HOURS}h  (absolute — zero inv is no exception)")

for p, (skip, reason) in skipped_parts.items():
    if skip:
        inv_note = f"  [inv=0]" if inventory.get(p, 0) == 0 else ""
        print(f"    SKIP  {p:30s}  {reason}{inv_note}")

# =============================================================
# SECTION 7B — CHANGEOVER TIMES  (VT only)
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

print(f"\n  VT changeover times loaded: {len(vt_changeover)} machines")
for m, h in vt_changeover.items():
    print(f"    {m:25s} → {h*60:.0f} min ({h:.3f} h)")

# =============================================================
# SECTION 7C — PART CATEGORY  (Runner / Repeater / Stranger)
# =============================================================

def build_category_from_sheet(df, sheet_name):
    cat     = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"),     None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        print(f"  WARNING: 'Part' column not found in '{sheet_name}'")
        return cat
    if cat_col is None:
        print(f"  WARNING: 'Category' column not found in '{sheet_name}' — all parts set to Stranger")
        # Default all parts to Stranger if column missing
        for _, row in df.iterrows():
            p = row[part_col]
            if pd.notna(p) and str(p).strip():
                cat[str(p).strip()] = "Stranger"
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        val  = str(row[cat_col]).strip() if pd.notna(row[cat_col]) else "Stranger"
        if pd.isna(part) or str(part).strip() == "":
            continue
        if val.lower() in ("runner", "repeater", "stranger"):
            val = val.capitalize()
        cat[str(part).strip()] = val
    return cat

part_category  = build_category_from_sheet(vt_parts_raw, "VT")
runner_count   = sum(1 for v in part_category.values() if v == "Runner")
repeater_count = sum(1 for v in part_category.values() if v == "Repeater")
stranger_count = sum(1 for v in part_category.values() if v == "Stranger")
print(f"  Part categories — Runner: {runner_count}  "
      f"Repeater: {repeater_count}  Stranger: {stranger_count}")

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        with open(MACHINE_STATE_FILE) as f:
            state = json.load(f)
        print(f"\n  Machine state loaded from '{MACHINE_STATE_FILE}':")
        print(f"  Changeover will be charged only where last-run part differs from today's plan.")
        for m, p in state.items():
            print(f"    {m:25s} last ran → {p}")
        return state

    # No state file — first run, clean slate
    print(f"\n  Machine state : FIRST RUN — no history file found.")
    print(f"  All machines start at 00:00 with no changeover today.")
    print(f"  Reason        : last-run part unknown → assume mould already loaded.")
    print(f"  After this run: state saved to '{MACHINE_STATE_FILE}'.")
    print(f"  From day 2    : changeover charged only where last part differs.")
    return {}

def save_machine_state(vt_state):
    combined = {m: p for m, p in vt_state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")
    print(f"  Tomorrow: changeover charged only where today's last part differs from plan.")
    for m, p in combined.items():
        print(f"    {m:25s} last ran → {p}")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX  (VT only)
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — INDENT HORIZON TABLE
# -------------------------------------------------------------
# Shows per-part planning numbers and skip/force status.
# =============================================================

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":              p,
            "Monthly_Indent":    round(monthly, 0),
            "Indent_Hrs_Total":  round(indent_hrs, 2),
            "Working_Days":      WORKING_DAYS,
            "Daily_Indent":      round(daily, 2),
            "Inventory_Now":     round(inv, 0),
            "Today_Target_Qty":  round(target, 0),
            "Today_Target_Hrs":  round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":       round(r, 2),
            "Indent_Status":     status,
            "Skip_Reason":       skip_reason,
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 11 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip:
            continue
        if daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — All active parts have zero indent today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < TARGET_DAYS_INV)

    if critical == n:
        return 0, "SCENARIO 0 — ALL parts critical (inv < 1 day indent)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical (inv < 1 day indent)"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, f"SCENARIO 3 — All parts healthy (≥{TARGET_DAYS_INV} days inv)"

# =============================================================
# SECTION 12 — PRIORITY ORDERING
# -------------------------------------------------------------
# Primary sort  : Category tier — Runner (0) > Repeater (1) > Stranger (2)
#                 Runners are always planned first, no Runner should
#                 ever be displaced by a Repeater or Stranger.
# Secondary sort: daily_indent descending within each category tier
#                 — highest requirement gets machine time first.
#
# "Inventory sufficient" parts are excluded before this is called.
# =============================================================

CATEGORY_TIER = {"Runner": 0, "Repeater": 1, "Stranger": 2}

def compute_priority(parts, horizon_df):
    horizon = horizon_df.set_index("Part")
    rows    = []
    for p in parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        days_cov = inv / daily if daily > 0 else 999
        target   = horizon.loc[p, "Today_Target_Qty"] if p in horizon.index else 0
        cat      = part_category.get(p, "Stranger")
        tier     = CATEGORY_TIER.get(cat, 2)

        rows.append({
            "Part":           p,
            "Category":       cat,
            "Tier":           tier,
            "Days_Coverage":  round(days_cov, 2),
            "Inv_Now":        round(inv, 0),
            "Monthly_Indent": round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":   round(daily, 2),
            "Today_Target":   round(target, 0),
            "Zero_Inv":       "YES" if inv == 0 else "No",
        })

    return (pd.DataFrame(rows)
              .sort_values(["Tier", "Daily_Indent"],
                           ascending=[True, False])   # tier asc, indent desc
              .drop(columns=["Tier"])
              .reset_index(drop=True))

# =============================================================
# SECTION 13 — MACHINE RANKER
# =============================================================

def rank_machines(part, machines, compatibility, machine_hours,
                  machine_last_part, changeover_dict, inv_days):
    category    = part_category.get(part, "Stranger")
    is_runner   = (category == "Runner")
    runner_lock = is_runner and inv_days <= 1.0

    ranked = []
    for m in machines:
        if m not in compatibility.get(part, []):
            continue
        used = machine_hours.get(m, 0)
        free = AVAILABLE_HOURS - used
        if free < MIN_RUN_HOURS:
            continue
        last   = machine_last_part.get(m)   # None = unknown (first run / clean slate)
        if runner_lock and last != part:
            continue
        # Changeover rules:
        #   last == part  → same mould loaded → NO changeover
        #   last == None  → unknown history (first run, clean slate) → NO changeover
        #   last != part  → different part was running → changeover required
        if last is None or last == part:
            co_hrs = 0.0
        else:
            co_hrs = changeover_dict.get(m, DEFAULT_CHANGEOVER_HRS)
        effective_free = free - co_hrs
        if effective_free < MIN_RUN_HOURS:
            continue
        cost = co_hrs / AVAILABLE_HOURS + (used / AVAILABLE_HOURS)
        ranked.append((m, cost, effective_free, co_hrs))

    ranked.sort(key=lambda x: x[1])
    return ranked, runner_lock

# =============================================================
# SECTION 14 — CSP TOOL-CHANGER SCHEDULER
# -------------------------------------------------------------
# ONE tool changer serves ALL 30 VT machines.
# HARD CONSTRAINT: no two changeover intervals may overlap.
#   i.e. if CO_i starts at S_i and takes D_i hours,
#        then for all i≠j: S_j >= S_i + D_i  OR  S_i >= S_j + D_j
#
# OBJECTIVE: Never let a machine sit idle.
#   When a machine must wait for the tool changer, extend the
#   run of the part BEFORE the CO to fill the waiting time
#   → machine keeps producing, zero idle time.
#
# APPROACH — Interval Scheduling CSP:
#
#   STEP 1  COLLECT
#     Extract every CO event from the plan. Each event has:
#       machine, part_before, part_after, co_duration,
#       natural_start (when the CO would happen with no conflicts),
#       row_before / row_after (plan row references for in-place edit)
#
#   STEP 2  CSP — find optimal CO ordering
#     Variables  : one integer index per CO event = its position
#                  in the tool-changer queue (0-based)
#     Constraint : the assigned time slots must not overlap
#     Method     : constraint-propagation + backtracking
#                  (python-constraint library)
#                  Falls back to greedy sort if library unavailable.
#
#     The CSP works on a DISCRETIZED timeline (1-min slots) to
#     keep the search space finite and fast for 20–35 events.
#
#   STEP 3  ASSIGN ACTUAL START TIMES
#     Walk the ordered queue. For each CO:
#       actual_start = max(natural_start, tool_changer_free_at)
#       If actual_start > natural_start → machine must wait.
#         Extend row_before run by the wait duration.
#         extra_qty = wait_hrs × rate  (free production)
#       tool_changer_free_at = actual_start + co_duration
#
#   STEP 4  PRINT
#     Full queue table + summary statistics.
#
# INSTALL (once, before first run):
#   pip install python-constraint
# =============================================================

def _fmt_h(h):
    """Convert decimal hours to HH:MM string."""
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"


def _collect_co_events(plan, machines):
    """
    Extract all CO events from the plan.
    Returns list of dicts, one per changeover.
    natural_start is recomputed live from current plan row values
    so it reflects any prior edits.
    """
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":      m,
                    "part_before":  m_rows[i - 1]["Part"],
                    "part_after":   row["Part"],
                    "co_duration":  co_h,
                    "natural_start": cursor,
                    "row_before":   m_rows[i - 1],
                    "row_after":    row,
                    # filled in Step 3
                    "actual_start": None,
                    "wait_hrs":     0.0,
                })
            cursor += co_h + run_h
    return events


def _recompute_natural_start(ev, plan):
    """
    Recompute natural_start for an event from current plan row values.
    (row_before may have been extended by a previous iteration.)
    """
    m       = ev["machine"]
    target  = ev["row_after"]
    m_rows  = [r for r in plan if r["Machine"] == m]
    cursor  = 0.0
    for r in m_rows:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor


def _machine_spare(m, plan):
    """Remaining hours on machine m before hitting 22h ceiling."""
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)


def _extend_row_before(ev, wait_hrs, plan):
    """
    Extend the run of the part BEFORE this CO by wait_hrs.
    Returns (actually_extended_hrs, extra_qty).
    Capped at machine's remaining spare capacity.
    """
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0

    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)

    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = (
        f"CSP: extended +{round(extend_by*60,1)}min to fill TC wait"
        + (f"  ⚠ {round((wait_hrs-extend_by)*60,1)}min unresolvable (22h ceiling)"
           if extend_by < wait_hrs - 0.001 else "")
    )
    return extend_by, extra


def _csp_order_events(events):
    """
    Use python-constraint to find the optimal ordering of CO events
    such that no two CO intervals overlap on the tool-changer timeline.

    Strategy:
      - Discretize timeline into 1-minute slots (0..1319 for 22h).
      - Variable per event = start slot in tool-changer queue.
      - Constraint: for every pair (i,j), their intervals must not overlap.
      - Minimize: sum of start slots (get COs done as early as possible
                  so machines wait the least before the no-idle fill).

    Returns: list of events in the CSP-optimal order.

    If python-constraint is not installed, falls back to greedy
    (sort by natural_start ascending) with a clear warning.
    """
    n = len(events)
    if n == 0:
        return events

    # ── Try python-constraint ────────────────────────────────
    try:
        from constraint import Problem, AllDifferentConstraint

        SLOTS = int(AVAILABLE_HOURS * 60)   # 1320 slots for 22h

        problem = Problem()

        # Variable for each event: start time in minutes (integer)
        for i, ev in enumerate(events):
            # Domain: earliest possible = 0, latest = ceiling - duration
            dur_min = int(math.ceil(ev["co_duration"] * 60))
            lo      = 0
            hi      = max(0, SLOTS - dur_min)
            problem.addVariable(i, range(lo, hi + 1))

        # Hard constraint: no two CO intervals overlap
        def no_overlap(s_i, s_j, dur_i, dur_j):
            # i ends before j starts  OR  j ends before i starts
            return (s_i + dur_i <= s_j) or (s_j + dur_j <= s_i)

        for i in range(n):
            for j in range(i + 1, n):
                dur_i = int(math.ceil(events[i]["co_duration"] * 60))
                dur_j = int(math.ceil(events[j]["co_duration"] * 60))
                # Capture dur_i, dur_j in closure
                def make_constraint(di, dj):
                    return lambda si, sj: (si + di <= sj) or (sj + dj <= si)
                problem.addConstraint(make_constraint(dur_i, dur_j), (i, j))

        print(f"    CSP: solving for {n} CO events "
              f"({n*(n-1)//2} no-overlap constraints) ...")

        solutions = problem.getSolutions()

        if not solutions:
            print("    CSP: no solution found — falling back to greedy sort")
            return sorted(events, key=lambda e: e["natural_start"])

        # Pick solution that minimises sum of assigned start times
        # (gets COs done earliest → least machine waiting)
        best = min(solutions, key=lambda sol: sum(sol.values()))

        # Re-order events by their assigned start slot
        ordered = sorted(range(n), key=lambda i: best[i])
        print(f"    CSP: optimal solution found  "
              f"(makespan = {max(best[i] + int(math.ceil(events[i]['co_duration']*60)) for i in range(n))} min)")
        return [events[i] for i in ordered]

    except ImportError:
        print("    WARNING: python-constraint not installed.")
        print("    Run:  pip install python-constraint")
        print("    Falling back to greedy sort by natural_start.")
        return sorted(events, key=lambda e: e["natural_start"])

    except Exception as exc:
        print(f"    WARNING: CSP solver error ({exc}) — falling back to greedy sort.")
        return sorted(events, key=lambda e: e["natural_start"])


def stagger_changeovers(plan, machines, changeover_dict):
    import math

    print(f"\n  {'='*62}")
    print(f"  CSP TOOL-CHANGER SCHEDULER  (1 tool changer — all VT machines)")
    print(f"  {'='*62}")
    print(f"  Hard constraint : no two CO intervals may overlap")
    print(f"  Objective       : extend run before CO to fill any machine wait")

    # ── STEP 1: collect ──────────────────────────────────────
    events = _collect_co_events(plan, machines)

    if not events:
        print("  No changeovers in plan — tool changer idle all shift  ✓")
        return

    print(f"\n  CO events found : {len(events)} across {len({e['machine'] for e in events})} machines")

    # ── STEP 2: CSP — find optimal ordering ──────────────────
    print(f"\n  STEP 2 — CSP ordering:")
    ordered_events = _csp_order_events(events)

    # ── STEP 3: assign actual start times & extend runs ──────
    print(f"\n  STEP 3 — Assign actual start times & extend runs to fill wait:")
    print(f"\n  {'#':<4} {'Machine':<18} {'Part Before':<24} {'Part After':<24} "
          f"{'CO':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7} {'Extra Pcs':>9}")
    print(f"  {'─'*4} {'─'*18} {'─'*24} {'─'*24} "
          f"{'─'*5} {'─'*8} {'─'*8} {'─'*7} {'─'*9}")

    tool_changer_free_at = 0.0
    total_wait_min       = 0.0
    total_extra_pcs      = 0
    unresolvable_count   = 0

    for idx, ev in enumerate(ordered_events, 1):
        # Recompute natural_start from live plan (may have shifted
        # due to extensions applied to earlier events on same machine)
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]

        # Actual start = latest of (when machine is ready, when TC is free)
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = actual_start - natural_start   # how long machine must wait

        # Extend run before CO to fill the wait — machine never idles
        extended_hrs, extra_pcs = 0.0, 0
        if wait_hrs > 0.001:
            extended_hrs, extra_pcs = _extend_row_before(ev, wait_hrs, plan)
            shortfall = wait_hrs - extended_hrs
            if shortfall > 0.001:
                unresolvable_count += 1

        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs

        tool_changer_free_at = actual_start + co_h
        total_wait_min      += wait_hrs * 60
        total_extra_pcs     += extra_pcs

        # Print row
        wait_str  = f"+{round(wait_hrs*60,1)}m" if wait_hrs > 0.001 else "none"
        extra_str = f"+{extra_pcs:.0f}" if extra_pcs > 0 else "—"
        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<24} "
              f"{ev['part_after']:<24} "
              f"{round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7} "
              f"{extra_str:>9}")

    # ── STEP 4: summary ───────────────────────────────────────
    events_delayed = sum(1 for e in ordered_events if e["wait_hrs"] > 0.001)

    print(f"\n  {'─'*62}")
    print(f"  CSP Tool-Changer Summary:")
    print(f"    Total CO events          : {len(ordered_events)}")
    print(f"    Events requiring delay   : {events_delayed}")
    print(f"    Total machine wait filled: {round(total_wait_min,1)} min  "
          f"(converted to production)")
    print(f"    Extra pieces gained      : {total_extra_pcs:,.0f}  "
          f"(from extended runs)")
    print(f"    Unresolvable waits (22h) : {unresolvable_count}"
          + ("  ← machine hits ceiling, small gap remains" if unresolvable_count else "  ✓"))
    print(f"    Tool changer finishes at : {_fmt_h(tool_changer_free_at)}")

    # Final serialized queue
    print(f"\n  Final tool-changer queue (CSP-optimal order):")
    print(f"  {'Slot':<5} {'Machine':<18} {'CO Start':>8} {'CO End':>8} "
          f"{'Duration':>9} {'Part → Part'}")
    print(f"  {'─'*5} {'─'*18} {'─'*8} {'─'*8} {'─'*9} {'─'*40}")
    for idx, ev in enumerate(ordered_events, 1):
        s   = ev["actual_start"]
        dur = ev["co_duration"]
        print(f"  {idx:<5} {ev['machine']:<18} {_fmt_h(s):>8} "
              f"{_fmt_h(s+dur):>8} {round(dur*60,0):>7.0f}min  "
              f"{ev['part_before']} → {ev['part_after']}")
    print(f"  {'='*62}")

# =============================================================
# SECTION 14B — 22-HOUR FILLER
# =============================================================

def fill_remaining_hours(plan, machine_hours, machine_last_part,
                         parts, compatibility, machines,
                         current_inventory, horizon_df):
    already_planned = {row["Part"] for row in plan}

    for m in machines:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
        if remaining <= 0:
            continue

        on_machine = [row["Part"] for row in plan if row["Machine"] == m]

        # ── Check if any part still has unmet daily indent ───
        # If yes, skip Option A entirely — free time must go to
        # a new part (Option B) to meet that part's daily indent first.
        unmet_parts = [
            p for p in parts
            if not should_skip(p)[0]
            and p not in already_planned
            and m in compatibility.get(p, [])
            and indent_daily.get(p, 0) > 0
            and current_inventory.get(p, 0) < indent_daily.get(p, 0)
        ]
        has_unmet = len(unmet_parts) > 0

        # Option A — extend existing part on this machine.
        # Only done when NO other part on this machine has an unmet
        # daily indent. Cap extension at MAX_OVERPRODUCTION_DAYS.
        if not has_unmet:
            best_part, best_gap = None, -1
            for p in on_machine:
                daily_p  = indent_daily.get(p, 0)
                produced = sum(r["Production_Qty"] for r in plan if r["Part"] == p)
                inv_p    = inventory.get(p, 0)
                # Headroom = how many more units before hitting 5-day cap
                headroom = max(0.0,
                               MAX_OVERPRODUCTION_DAYS * daily_p - inv_p - produced)
                if headroom > best_gap:
                    best_gap, best_part = headroom, p

            if best_part and best_gap > 0:
                p              = best_part
                r_val          = rate.get(p, 1)
                hrs_to_cap     = best_gap / r_val if r_val > 0 else 0
                extend_hrs     = min(remaining, hrs_to_cap)
                if extend_hrs >= 0.05:
                    extra_qty = round(extend_hrs * r_val, 0)
                    for row in plan:
                        if row["Part"] == p and row["Machine"] == m:
                            row["Run_Hours"]      = round(row["Run_Hours"] + extend_hrs, 2)
                            row["Production_Qty"] = round(row["Production_Qty"] + extra_qty, 0)
                            row["Total_Hrs_Used"] = round(
                                row.get("Total_Hrs_Used", row["Run_Hours"]) + extend_hrs, 2)
                            row["Type"] = "Primary+Extended"
                            break
                    machine_hours[m]    += extend_hrs
                    current_inventory[p] = current_inventory.get(p, 0) + extra_qty
                    remaining            = round(remaining - extend_hrs, 3)
                    print(f"    ↑ {p:30s} → {m:15s}  +{extend_hrs:.2f}h  "
                          f"qty+={extra_qty:.0f}  [EXTEND ≤ {MAX_OVERPRODUCTION_DAYS}d cap]")

        # Option B — add a new part (≥ MIN_RUN_HOURS required)
        if remaining < MIN_RUN_HOURS:
            continue

        candidates = []
        for p in parts:
            if p in already_planned:
                continue
            if m not in compatibility.get(p, []):
                continue
            skip, _ = should_skip(p)
            if skip:
                continue   # skip rule is absolute — never added even in filler
            daily   = indent_daily.get(p, 0)
            inv_now  = current_inventory.get(p, 0)
            # Skip if inventory already sufficient
            if inv_now >= daily > 0:
                continue
            candidates.append((p, daily))

        # Sort by daily_indent descending — biggest parts fill first
        candidates.sort(key=lambda x: -x[1])

        for p, daily_val in candidates:
            last   = machine_last_part.get(m)
            co_hrs = 0.0 if (last is None or last == p) else vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            eff_run = remaining - co_hrs
            if eff_run < MIN_RUN_HOURS:
                continue
            r_val            = rate.get(p, 1)
            # Cap: never produce beyond MAX_OVERPRODUCTION_DAYS × daily_indent
            inv_p            = current_inventory.get(p, 0)
            daily_p          = indent_daily.get(p, 0)
            headroom_qty     = max(0.0, MAX_OVERPRODUCTION_DAYS * daily_p - inv_p)
            hrs_for_headroom = headroom_qty / r_val if r_val > 0 else eff_run
            eff_run          = min(eff_run, hrs_for_headroom)
            if eff_run < MIN_RUN_HOURS:
                continue
            qty = round(eff_run * r_val, 0)
            machine_hours[m]    += (co_hrs + eff_run)
            current_inventory[p] = current_inventory.get(p, 0) + qty
            machine_last_part[m] = p
            already_planned.add(p)
            plan.append({
                "Part":             p,
                "Category":         part_category.get(p, "Stranger"),
                "Machine":          m,
                "Run_Hours":        round(eff_run, 2),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + eff_run, 2),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(indent_daily.get(p, 0), 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Type":             "New — filler",
                "Runner_Lock":      "No",
                "Priority_Score":   0,
                "Stagger_Adjusted": "No",
            })
            print(f"    + {p:30s} → {m:15s}  {eff_run:.2f}h  qty={qty:.0f}  "
                  f"[Filler]  CO={'Yes' if co_hrs > 0 else 'No'}")
            break

# =============================================================
# SECTION 15 — MAIN SCHEDULER
# =============================================================

def schedule(parts, compatibility, machines, changeover_dict, label=""):

    print(f"\n{'─'*62}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(machines)} machines")
    print(f"{'─'*62}")

    scenario, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    horizon_df = compute_indent_horizon(parts)

    # Print what needs production
    prod_needed = horizon_df[
        horizon_df["Indent_Status"].isin(["PRODUCTION NEEDED", "ZERO INV — FORCED"])
    ]
    if not prod_needed.empty:
        print(f"\n  Parts requiring production today ({len(prod_needed)}):")
        print(f"  {'Part':<30}  {'Monthly':>8}  {'Daily':>7}  {'Inv':>8}  "
              f"{'Target':>8}  {'Status'}")
        print(f"  {'─'*30}  {'─'*8}  {'─'*7}  {'─'*8}  {'─'*8}  {'─'*20}")
        for _, r in prod_needed.iterrows():
            print(f"  {r['Part']:<30}  "
                  f"{r['Monthly_Indent']:>8.0f}  "
                  f"{r['Daily_Indent']:>7.2f}  "
                  f"{r['Inventory_Now']:>8.0f}  "
                  f"{r['Today_Target_Qty']:>8.0f}  "
                  f"{r['Indent_Status']}")
    else:
        print("  No production needed — inventory covers all active parts today")

    machine_hours     = {m: 0.0 for m in machines}
    machine_last_part = {m: machine_state.get(m) for m in machines}
    current_inventory = inventory.copy()

    plan, deferred, not_planned, skipped_log = [], [], [], []

    # Active parts = passed skip rules AND inventory insufficient.
    # Skip rules are absolute — zero inventory is no exception.
    active_parts = [
        p for p in parts
        if not should_skip(p)[0]
        and indent_monthly.get(p, 0) > 0
        and not (inventory.get(p, 0) >= indent_daily.get(p, 0) > 0)
    ]

    priority_df = compute_priority(active_parts, horizon_df)

    print(f"\n  Priority order — Runner > Repeater > Stranger, then daily_indent desc "
          f"({len(priority_df)} active parts):")
    print(f"  {'Part':<30}  {'Cat':<10}  {'Daily':>7}  {'Inv':>8}  {'Target':>8}  {'Zero_Inv':>8}")
    print(f"  {'─'*30}  {'─'*10}  {'─'*7}  {'─'*8}  {'─'*8}  {'─'*8}")
    for _, r in priority_df.iterrows():
        print(f"  {r['Part']:<30}  {r['Category']:<10}  "
              f"{r['Daily_Indent']:>7.2f}  "
              f"{r['Inv_Now']:>8.0f}  "
              f"{r['Today_Target']:>8.0f}  "
              f"{r['Zero_Inv']:>8}")

    runner_dedicated_machines = set()

    # ── Log skipped parts (not zero-inv) ─────────────────────
    for p in parts:
        skip, reason = should_skip(p)
        inv_now = inventory.get(p, 0)
        if skip and inv_now > 0:
            skipped_log.append({
                "Part":           p,
                "Category":       part_category.get(p, "Stranger"),
                "Rate_Per_Hour":  round(rate.get(p, 0), 2),
                "Monthly_Indent": round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":   round(indent_daily.get(p, 0), 2),
                "Inventory_Now":  round(inv_now, 0),
                "Skip_Reason":    reason,
            })

    def run_assignment(parts_subset, pass_label, exclude_machines=None):
        if exclude_machines is None:
            exclude_machines = set()
        print(f"\n  {pass_label}:")

        for _, row in priority_df[priority_df["Part"].isin(parts_subset)].iterrows():
            part     = row["Part"]

            if any(r["Part"] == part for r in plan):
                print(f"    ~ {part:30s}  ALREADY PLANNED — skipped")
                continue

            inv_now  = current_inventory.get(part, 0)
            daily    = indent_daily.get(part, 0)
            monthly  = indent_monthly.get(part, 0)
            r_val    = rate.get(part, 1)
            category = part_category.get(part, "Stranger")
            inv_days = inv_now / daily if daily > 0 else 999
            target   = today_target_qty.get(part, 0)

            compatible_mch = compatibility.get(part, [])

            # ── Zero indent ───────────────────────────────────
            if monthly == 0:
                deferred.append({
                    "Part":           part,
                    "Category":       category,
                    "Rate_Per_Hour":  round(r_val, 2),
                    "Inventory_Now":  round(inv_now, 0),
                    "Monthly_Indent": 0,
                    "Daily_Indent":   0,
                    "Today_Target":   0,
                    "Days_Coverage":  round(inv_days, 2),
                    "Reason":         "Monthly indent is zero",
                    "Next_Action":    "Re-evaluate when indent is set",
                })
                print(f"    - {part:30s} [{category:8s}]  DEFERRED — indent = 0")
                continue

            # ── Inventory sufficient (and not zero-inv forced) ─
            if target == 0 and inv_now > 0:
                deferred.append({
                    "Part":           part,
                    "Category":       category,
                    "Rate_Per_Hour":  round(r_val, 2),
                    "Inventory_Now":  round(inv_now, 0),
                    "Monthly_Indent": round(monthly, 0),
                    "Daily_Indent":   round(daily, 2),
                    "Today_Target":   0,
                    "Days_Coverage":  round(inv_days, 2),
                    "Reason":         "Inventory covers daily indent — no production needed",
                    "Next_Action":    "Re-evaluate tomorrow",
                })
                print(f"    - {part:30s} [{category:8s}]  DEFERRED  "
                      f"inv={inv_now:.0f} ≥ daily={daily:.2f}")
                continue

            # ── Target hours ──────────────────────────────────
            # Phase 1 — Primary target:
            #   Produce exactly today_target = max(0, daily - inv)
            #   This is what the part genuinely needs today.
            #
            # Phase 2 — Over-production ceiling:
            #   Total stock after today (inv + produced) must not
            #   exceed MAX_OVERPRODUCTION_DAYS × daily_indent.
            #   headroom = (MAX_OVERPRODUCTION_DAYS × daily) - inv_now
            #   target_hrs = min(primary_hrs, headroom_hrs)
            #
            # Floor: MIN_RUN_HOURS (never assign less than a viable block)
            primary_qty    = target                          # today_target
            headroom_qty   = max(0.0,
                                 MAX_OVERPRODUCTION_DAYS * daily - inv_now)
            capped_qty     = min(primary_qty, headroom_qty)
            target_hrs     = max(
                MIN_RUN_HOURS,
                min(capped_qty / r_val if r_val > 0 else MIN_RUN_HOURS,
                    AVAILABLE_HOURS)
            )

            # ── No compatible machines ────────────────────────
            if not compatible_mch:
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Rate_Per_Hour":       round(r_val, 2),
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": "NONE DEFINED",
                    "Reason":              "Part has no compatible machines in VT matrix",
                    "Action_Needed":       "Add part to VT_Matrix",
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — not in matrix")
                continue

            available_mch = [m for m in compatible_mch if m not in exclude_machines]
            if not available_mch:
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Rate_Per_Hour":       round(r_val, 2),
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              "All compatible machines dedicated to Runners",
                    "Action_Needed":       "Add more compatible machines in matrix",
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — machines Runner-dedicated")
                continue

            ranked, runner_lock = rank_machines(
                part, available_mch, compatibility,
                machine_hours, machine_last_part,
                changeover_dict, inv_days
            )

            if not ranked:
                free_map = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                            for m in available_mch if m in machine_hours}
                if runner_lock:
                    last_m = next(
                        (m for m in available_mch if machine_last_part.get(m) == part), None)
                    free_l = round(AVAILABLE_HOURS - machine_hours.get(last_m, 0), 2) if last_m else 0
                    reason = (f"RUNNER (inv≤1d) — must stay on {last_m}, "
                              f"only {free_l}h free") if last_m else \
                             "RUNNER — no machine state (first run)"
                    action = f"Free capacity on {last_m}" if last_m else \
                             "Machine state saves after this run"
                elif all(h < MIN_RUN_HOURS for h in free_map.values()):
                    reason = ("All compatible machines full. Free: "
                              + ", ".join(f"{m}={h}h" for m, h in free_map.items()))
                    action = "Reduce lower-priority part hours or plan tomorrow"
                else:
                    reason = ("Remaining hrs < MIN after changeover. Free: "
                              + ", ".join(f"{m}={h}h" for m, h in free_map.items()))
                    action = "Check Inventory_Health sheet"

                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Rate_Per_Hour":       round(r_val, 2),
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              reason,
                    "Action_Needed":       action,
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — {reason[:60]}")
                continue

            # ── Assign ────────────────────────────────────────
            assigned = False
            for m, cost, effective_free, co_hrs in ranked:
                run_h = min(target_hrs, effective_free)
                run_h = max(run_h, MIN_RUN_HOURS)
                run_h = min(run_h, effective_free)
                qty   = round(run_h * r_val, 0)

                machine_hours[m]        += (co_hrs + run_h)
                current_inventory[part]  = current_inventory.get(part, 0) + qty
                machine_last_part[m]     = part

                if category == "Runner":
                    runner_dedicated_machines.add(m)

                forced_tag = "ZERO-INV-FORCED" if inv_now == 0 else "Primary"
                plan.append({
                    "Part":              part,
                    "Category":          category,
                    "Machine":           m,
                    "Run_Hours":         round(run_h, 2),
                    "Changeover_Hrs":    round(co_hrs, 3),
                    "Total_Hrs_Used":    round(co_hrs + run_h, 2),
                    "Rate_Per_Hour":     round(r_val, 2),
                    "Production_Qty":    qty,
                    "Monthly_Indent":    round(monthly, 0),
                    "Daily_Indent":      round(daily, 2),
                    "Today_Target":      round(target, 0),
                    "Changeover":        "No" if co_hrs == 0 else "Yes",
                    "Type":              forced_tag,
                    "Runner_Lock":       "YES" if runner_lock else "No",
                    "Priority_Score":    round(indent_daily.get(part, 0), 2),
                    "Stagger_Adjusted":  "No",
                })
                co_str = "No" if co_hrs == 0 else f"Yes ({co_hrs*60:.0f}min)"
                tag    = "  [DEDICATED]" if category == "Runner" else \
                         "  [ZERO-INV FORCED]" if inv_now == 0 else ""
                print(f"    ✓ {part:30s} [{category:8s}] → {m:15s}  "
                      f"{run_h:.2f}h  qty={qty:>8.0f}  CO={co_str}{tag}")
                assigned = True
                break

            if not assigned:
                free_map = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                            for m in available_mch if m in machine_hours}
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "Rate_Per_Hour":       round(r_val, 2),
                    "Monthly_Indent":      round(monthly, 0),
                    "Daily_Indent":        round(daily, 2),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Today_Target":        round(target, 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              (
                        f"No single machine has {round(target_hrs,2)}h free after CO. "
                        + "Free: " + ", ".join(f"{m}={h}h" for m, h in free_map.items())
                    ),
                    "Action_Needed": "Reduce lower-priority hours or verify inventory covers gap",
                })
                print(f"    ✗ {part:30s} [{category:8s}]  NOT PLANNED — "
                      f"no machine has {round(target_hrs,2)}h free")

    # PASS 1 — Runners
    runner_parts     = [p for p in parts
                        if part_category.get(p, "Stranger") == "Runner"
                        and not should_skip(p)[0]]
    non_runner_parts = [p for p in parts
                        if part_category.get(p, "Stranger") != "Runner"
                        and not should_skip(p)[0]]

    run_assignment(runner_parts,
                   "PASS 1 — Runners (dedicated machine, cap = target + 1-day buffer)")
    print(f"\n  Runner-dedicated machines: "
          f"{sorted(runner_dedicated_machines) if runner_dedicated_machines else 'none'}")

    # PASS 2 — Repeaters + Strangers
    run_assignment(non_runner_parts,
                   "PASS 2 — Repeaters + Strangers (Runner machines excluded)",
                   exclude_machines=runner_dedicated_machines)

    # 22-hour filler
    print(f"\n  22-hour filler:")
    fill_remaining_hours(
        plan, machine_hours, machine_last_part,
        list(parts), compatibility, machines,
        current_inventory, horizon_df
    )

    # Duplicate check
    part_counts = {}
    for row in plan:
        part_counts[row["Part"]] = part_counts.get(row["Part"], 0) + 1
    dupes = {p: c for p, c in part_counts.items() if c > 1}
    if dupes:
        print(f"  WARNING: duplicate parts: {dupes}")
    else:
        print(f"    No duplicate parts — each part appears exactly once  ✓")

    # Stagger
    stagger_changeovers(plan, machines, changeover_dict)

    # ── Daily indent status — planned parts only ─────────────
    # For every planned part: does today's production (planned qty)
    # meet the daily indent requirement by itself?
    # Note: inventory is NOT counted here — this shows purely
    # whether the machine run produces enough for one day's need.
    indent_status_rows = []
    for row in plan:
        p         = row["Part"]
        planned   = float(row.get("Production_Qty") or 0)
        daily     = indent_daily.get(p, 0)
        monthly   = indent_monthly.get(p, 0)
        r_val     = float(row.get("Rate_Per_Hour") or rate.get(p, 0))
        m         = row.get("Machine", "—")
        run_h     = float(row.get("Run_Hours") or 0)
        inv_b     = inventory.get(p, 0)

        # Gap = how many units short of daily indent (negative = surplus)
        gap       = daily - planned
        meets     = planned >= daily

        # Hours needed to produce exactly daily_indent from scratch
        hrs_needed = daily / r_val if r_val > 0 else 0

        indent_status_rows.append({
            "Part":               p,
            "Category":           part_category.get(p, "Stranger"),
            "Machine":            m,
            "Rate_Per_Hour":      round(r_val, 2),
            "Run_Hours":          round(run_h, 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Monthly_Indent":     round(monthly, 0),
            "Gap_vs_Daily":       round(gap, 0),   # +ve = short, -ve = surplus
            "Hrs_Needed_For_Daily": round(hrs_needed, 2),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),  # inv + produced
            "Covers_Daily_With_Inv": "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
        })

    # Sort: parts that miss daily indent first (most urgent at top),
    # then by gap descending within each group
    indent_status_df = pd.DataFrame(indent_status_rows) if indent_status_rows else pd.DataFrame()
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"],
            ascending=[True, False]   # NO first, then largest gap first
        ).reset_index(drop=True)

        # Print summary to console
        meets_count  = (indent_status_df["Meets_Daily_Indent"] == "YES ✓").sum()
        misses_count = (indent_status_df["Meets_Daily_Indent"] == "NO ✗").sum()
        covers_count = (indent_status_df["Covers_Daily_With_Inv"] == "YES ✓").sum()
        print(f"\n  Daily Indent Status (planned parts only):")
        print(f"    Parts meeting daily indent (production alone) : {meets_count}")
        print(f"    Parts missing daily indent (production alone) : {misses_count}")
        print(f"    Parts covered when inventory is added         : {covers_count}")
        if misses_count > 0:
            print(f"\n    Parts NOT meeting daily indent:")
            print(f"    {'Part':<30} {'Planned':>8} {'Daily':>8} {'Gap':>8} {'Hrs Needed':>10}")
            print(f"    {'─'*30} {'─'*8} {'─'*8} {'─'*8} {'─'*10}")
            for _, r in indent_status_df[
                indent_status_df["Meets_Daily_Indent"] == "NO ✗"
            ].iterrows():
                print(f"    {r['Part']:<30} {r['Planned_Qty']:>8.0f} "
                      f"{r['Daily_Indent']:>8.2f} {r['Gap_vs_Daily']:>8.0f} "
                      f"{r['Hrs_Needed_For_Daily']:>10.2f}h")
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        produced = sum(r["Production_Qty"] for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(monthly, 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Status":          ("OK"       if days_cov >= TARGET_DAYS_INV
                                else "LOW"      if days_cov >= 1
                                else "CRITICAL"),
        })

    # ── Machine utilization ───────────────────────────────────
    mach_rows = []
    for m in machines:
        used      = machine_hours.get(m, 0)
        parts_run  = [r["Part"] for r in plan if r["Machine"] == m]
        parts_rates = ", ".join(
            f"{p}:{round(rate.get(p,0),1)}"
            for p in parts_run
        )
        co_count  = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        util_status = ("FULL"     if used >= AVAILABLE_HOURS - 0.5 else
                       "GOOD"     if used >= AVAILABLE_HOURS * 0.85 else
                       "PARTIAL"  if used >= AVAILABLE_HOURS * 0.5 else
                       "UNDERUSED")
        mach_rows.append({
            "Machine":              m,
            "Total_Available_Hrs":  AVAILABLE_HOURS,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               util_status,
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
            "Part_Rates_Per_Hr":    parts_rates if parts_rates else "—",
        })

    plan_df     = pd.DataFrame(plan)         if plan         else pd.DataFrame()
    def_df      = pd.DataFrame(deferred)     if deferred     else pd.DataFrame()
    not_df      = pd.DataFrame(not_planned)  if not_planned  else pd.DataFrame()
    skip_df     = pd.DataFrame(skipped_log)  if skipped_log  else pd.DataFrame()
    mach_df     = pd.DataFrame(mach_rows)
    inv_df      = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df["Category"] = plan_df["Part"].map(
            lambda p: part_category.get(p, "Stranger"))
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)
        plan_df.insert(2, "Working_Days",  WORKING_DAYS)

    return plan_df, def_df, not_df, skip_df, mach_df, inv_df, machine_last_part, horizon_df, indent_status_df

# =============================================================
# SECTION 16 — RUN VT SCHEDULER
# =============================================================

vt_parts = data_valid[
    data_valid["Material"].isin(vt_matrix["Part"])
]["Material"].unique()

# =============================================================
# SECTION 16A — FULL PART AUDIT  (accounts for all VT parts)
# -------------------------------------------------------------
# Every part in the VT sheet is traced through every filter gate.
# Since VT sheet is now the single source of truth, gates are:
#
# GATE 1 — Cycle time / Cavity valid
#   Both must be present and non-zero to compute a rate.
#   Missing/zero → "Zero or missing Cycle time or Cavity in VT sheet"
#
# GATE 2 — In VT_Matrix
#   Part must appear in VT_Matrix with ≥1 machine assigned.
#   Missing → "Not in VT_Matrix — no compatible machine assigned"
#
# GATE 3 — Indent present
#   Monthly indent must be > 0.
#   Missing/zero → "Zero or missing Indent in VT sheet"
#
# GATE 4 — Skip rules  (bypassed when inventory = 0)
#   indent < 150  OR  indent / rate ≤ 4h → "Skipped"
#
# GATE 5 — Inventory sufficient
#   inventory ≥ daily_indent → "Not required today"
#
# PASSES ALL GATES → "ENTERS SCHEDULER"
#   Scheduler then decides: Planned / Not Planned (capacity)
# =============================================================

print(f"\n{'='*70}")
print(f"  PART AUDIT — tracing all {len(data)} VT parts through every filter gate")
print(f"{'='*70}")

# All parts from VT sheet (including those with missing rate)
all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []

for part in all_vt_parts_raw:
    inv     = inventory.get(part, 0.0)
    r_val   = rate.get(part, None)        # None if rate not computable
    monthly = indent_monthly.get(part, 0.0)
    daily   = indent_daily.get(part, 0.0)

    # Retrieve raw cycle time and cavity for display
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"

    # ── GATE 1: cycle time valid? ─────────────────────────────
    # Cavity no longer affects rate (cavities cancel in formula).
    # Only cycle time must be present and non-zero.
    if part in zero_rate_set or r_val is None:
        audit_rows.append({
            "Part":           part,
            "Gate_Failed":    "GATE 1",
            "Reason":         (
                f"Cycle time={ct_raw} — zero or missing in VT sheet "
                f"→ rate cannot be computed  (Cavity={cv_raw} — not used in rate)"
            ),
            "Monthly_Indent": round(monthly, 0),
            "Daily_Indent":   round(daily, 2),
            "Inventory":      round(inv, 0),
            "Rate_Per_Hour":    "—",
            "Cycle_Time":     ct_raw,
            "Cavity":         cv_raw,
            "Status":         "ZERO/MISSING CYCLE TIME",
        })
        continue

    # ── GATE 2: in VT_Matrix? ─────────────────────────────────
    if part not in matrix_parts:
        audit_rows.append({
            "Part":           part,
            "Gate_Failed":    "GATE 2",
            "Reason":         "Part not in VT_Matrix — no compatible machine assigned",
            "Monthly_Indent": round(monthly, 0),
            "Daily_Indent":   round(daily, 2),
            "Inventory":      round(inv, 0),
            "Rate_Per_Hour":    round(r_val, 2),
            "Cycle_Time":     ct_raw,
            "Cavity":         cv_raw,
            "Status":         "NOT IN VT_MATRIX",
        })
        continue

    # ── GATE 3: indent present? ───────────────────────────────
    if monthly == 0:
        audit_rows.append({
            "Part":           part,
            "Gate_Failed":    "GATE 3",
            "Reason":         "Monthly indent is zero or blank in VT sheet",
            "Monthly_Indent": 0,
            "Daily_Indent":   0,
            "Inventory":      round(inv, 0),
            "Rate_Per_Hour":    round(r_val, 2),
            "Cycle_Time":     ct_raw,
            "Cavity":         cv_raw,
            "Status":         "ZERO/MISSING INDENT",
        })
        continue

    # ── GATE 4: skip rules (ABSOLUTE — zero inventory is no exception) ──
    # Rule 1: daily_indent ≤ 150
    # Rule 2: whole monthly indent producible in ≤ 4h
    skip, skip_reason = should_skip(part)
    if skip:
        audit_rows.append({
            "Part":           part,
            "Gate_Failed":    "GATE 4",
            "Reason":         skip_reason + (
                "  [NOTE: inv=0 but still skipped — rule is absolute]"
                if inv == 0 else ""
            ),
            "Monthly_Indent": round(monthly, 0),
            "Daily_Indent":   round(daily, 2),
            "Inventory":      round(inv, 0),
            "Rate_Per_Hour":    round(r_val, 2),
            "Cycle_Time":     ct_raw,
            "Cavity":         cv_raw,
            "Status":         "SKIPPED (LOW INDENT / TRIVIAL RUN)",
        })
        continue

    # ── GATE 5: inventory sufficient? ────────────────────────
    if daily > 0 and inv >= daily:
        audit_rows.append({
            "Part":           part,
            "Gate_Failed":    "GATE 5",
            "Reason":         (
                f"Inventory ({round(inv,0):.0f}) >= "
                f"daily_indent ({round(daily,2):.2f}) — no production needed today"
            ),
            "Monthly_Indent": round(monthly, 0),
            "Daily_Indent":   round(daily, 2),
            "Inventory":      round(inv, 0),
            "Rate_Per_Hour":    round(r_val, 2),
            "Cycle_Time":     ct_raw,
            "Cavity":         cv_raw,
            "Status":         "NOT REQUIRED — INV SUFFICIENT",
        })
        continue

    # ── PASSED ALL GATES ─────────────────────────────────────
    audit_rows.append({
        "Part":           part,
        "Gate_Failed":    "—",
        "Reason":         "Passed all gates — enters scheduler",
        "Monthly_Indent": round(monthly, 0),
        "Daily_Indent":   round(daily, 2),
        "Inventory":      round(inv, 0),
        "Rate_Per_Hour":    round(r_val, 2),
        "Cycle_Time":     ct_raw,
        "Cavity":         cv_raw,
        "Status":         "ENTERS SCHEDULER",
    })

audit_df = pd.DataFrame(audit_rows)

# ── Print gate-level summary ──────────────────────────────────
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Gate breakdown ({len(all_vt_parts_raw)} total parts):")
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "✗"
    print(f"    {marker}  {status:<42} : {count:>4} parts")

# ── Print each dropped part with reason ──────────────────────
non_planned_audit = audit_df[audit_df["Gate_Failed"] != "—"]
if not non_planned_audit.empty:
    print(f"\n  Detail — {len(non_planned_audit)} parts dropped before scheduler:")
    print(f"  {'Part':<30}  {'Gate':<8}  {'Reason'}")
    print(f"  {'─'*30}  {'─'*8}  {'─'*55}")
    for _, r in non_planned_audit.sort_values("Gate_Failed").iterrows():
        print(f"  {r['Part']:<30}  {r['Gate_Failed']:<8}  {str(r['Reason'])[:55]}")

print(f"{'='*70}\n")

(vt_plan, vt_def, vt_not, vt_skip,
 vt_mach, vt_inv, vt_state, vt_horizon,
 vt_indent_status) = schedule(
    vt_parts, vt_compat, vt_machines, vt_changeover, "VT Machines"
)

save_machine_state(vt_state)

# =============================================================
# SECTION 17 — SAVE OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":       "0D6E6E",
    "VT_Plan":                  "1F4E79",
    "VT_Machine_Util":          "375623",
    "VT_Not_Planned":           "7B2C2C",
    "VT_Not_Required_Today":    "7F6000",
    "VT_Skipped_Parts":         "5C3D2E",
    "VT_Inventory_Health":      "4A235A",
    "VT_Indent_Horizon":        "154360",
    "VT_Part_Audit":            "1C3557",
    "VT_Daily_Indent_Status":   "0F4C2A",   # dark green — pass/fail sheet
}

STATUS_FILLS = {
    "FULL":                              PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":                              PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":                           PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":                         PatternFill("solid", fgColor="FFC7CE"),
    "OK":                                PatternFill("solid", fgColor="C6EFCE"),
    "LOW":                               PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":                          PatternFill("solid", fgColor="FFC7CE"),
    "PRODUCTION NEEDED":                 PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":                 PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":                    PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                           PatternFill("solid", fgColor="EDEDED"),
    "NO INDENT":                         PatternFill("solid", fgColor="EDEDED"),
    # Daily indent meet/miss
    "YES ✓":                             PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":                              PatternFill("solid", fgColor="FFC7CE"),
    "NOT REQUIRED — INV SUFFICIENT":     PatternFill("solid", fgColor="DDEBF7"),
    "SKIPPED (LOW INDENT / TRIVIAL RUN)":PatternFill("solid", fgColor="EDEDED"),
    "ZERO/MISSING CYCLE TIME":           PatternFill("solid", fgColor="FFC7CE"),
    "ZERO/MISSING CYCLE TIME OR CAVITY": PatternFill("solid", fgColor="FFC7CE"),  # legacy
    "NOT IN VT_MATRIX":                  PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":               PatternFill("solid", fgColor="FFEB9C"),
}

def style_sheet(ws, header_hex):
    hf = PatternFill("solid", fgColor=header_hex)
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(50, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and ("Status" in str(col_name) or "Indent_Status" in str(col_name)):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def build_machine_wise_plan(plan_df, machines):
    """
    Build the machine-wise sequenced plan with Rate_Per_Hour column included.
    One row per part per machine, plus a summary footer row per machine.
    """
    if plan_df.empty:
        return pd.DataFrame()

    def safe_float(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    # Use the global _fmt_h helper (defined alongside stagger_changeovers)
    fmt_time = _fmt_h

    rows = []
    for m in machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue

        cumulative_hrs = 0.0
        seq            = 1

        for _, pr in machine_rows.iterrows():
            part    = pr.get("Part", "—")
            run_h   = safe_float(pr.get("Run_Hours", 0))
            co_h    = safe_float(pr.get("Changeover_Hrs", 0))
            r_val   = safe_float(pr.get("Rate_Per_Hour", 0))
            co_mins = round(co_h * 60, 1)
            start_h = cumulative_hrs + co_h
            end_h   = start_h + run_h

            rows.append({
                "Machine":                m,
                "Seq":                    seq,
                "Part":                   part,
                "Category":               part_category.get(part, "Stranger"),
                "Rate_Per_Hour":          round(r_val, 2),          # ← added
                "Changeover_Before_Mins": co_mins,
                "Run_Hours":              round(run_h, 2),
                "Start_Time":             fmt_time(cumulative_hrs),
                "Start_After_CO":         fmt_time(start_h),
                "End_Time":               fmt_time(end_h),
                "Cumulative_Hrs":         round(end_h, 2),
                "Production_Qty":         safe_float(pr.get("Production_Qty", 0)),
                "Monthly_Indent":         safe_float(pr.get("Monthly_Indent", 0)),
                "Daily_Indent":           safe_float(pr.get("Daily_Indent", 0)),
                "Today_Target":           safe_float(pr.get("Today_Target", 0)),
                "Changeover":             pr.get("Changeover", "No") or "No",
                "Type":                   pr.get("Type", "Primary") or "Primary",
                "Runner_Lock":            pr.get("Runner_Lock", "No") or "No",
                "Row_Type":               "Part",
            })
            cumulative_hrs = end_h
            seq           += 1

        # Summary footer
        total_used     = round(cumulative_hrs, 2)
        co_hrs_series  = machine_rows["Changeover_Hrs"].apply(lambda x: safe_float(x, 0))
        total_co_mins  = round(co_hrs_series.sum() * 60, 1)
        total_prod_hrs = round(total_used - co_hrs_series.sum(), 2)
        total_qty      = machine_rows["Production_Qty"].apply(lambda x: safe_float(x, 0)).sum()
        co_count       = int(machine_rows["Changeover"].eq("Yes").sum())

        rows.append({
            "Machine":                m,
            "Seq":                    "—",
            "Part":                   f"TOTAL — {m}",
            "Category":               "—",
            "Rate_Per_Hour":          "—",
            "Changeover_Before_Mins": total_co_mins,
            "Run_Hours":              total_prod_hrs,
            "Start_Time":             "00:00",
            "Start_After_CO":         "—",
            "End_Time":               fmt_time(total_used),
            "Cumulative_Hrs":         total_used,
            "Production_Qty":         round(total_qty, 0),
            "Monthly_Indent":         "—",
            "Daily_Indent":           "—",
            "Today_Target":           "—",
            "Changeover":             f"{co_count} changeovers",
            "Type":                   f"Used {total_used}h / {AVAILABLE_HOURS}h  |  Unused {round(AVAILABLE_HOURS-total_used,2)}h",
            "Runner_Lock":            "—",
            "Row_Type":               "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    header_font  = Font(bold=True, color="FFFFFF", size=11)
    summary_fill = PatternFill("solid", fgColor="0D9488")
    summary_font = Font(bold=True, color="FFFFFF", size=11)
    part_fills   = [
        PatternFill("solid", fgColor="EFF6FF"),
        PatternFill("solid", fgColor="F0FDF4"),
    ]
    co_fill = PatternFill("solid", fgColor="FEF9C3")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers      = [cell.value for cell in ws[1]]
    row_type_col = headers.index("Row_Type")  + 1 if "Row_Type"  in headers else None
    co_col       = headers.index("Changeover") + 1 if "Changeover" in headers else None
    machine_col  = headers.index("Machine")   + 1 if "Machine"   in headers else None

    machine_color_idx = 0
    current_machine   = None

    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col - 1].value if row_type_col else ""
        machine  = row[machine_col  - 1].value if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = summary_font
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            bg = part_fills[machine_color_idx]
            for cell in row:
                cell.fill      = bg
                cell.alignment = Alignment(vertical="center")
            if co_col and row[co_col - 1].value == "Yes":
                row[co_col - 1].fill = co_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(40, max_len + 3))
    ws.freeze_panes = "C2"


# =============================================================
# SECTION 17A — WRITE EXCEL
# =============================================================

print(f"\nWriting → {output_path}")

vt_mw = build_machine_wise_plan(vt_plan, vt_machines)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    vt_mw.to_excel(writer,             sheet_name="VT_Plan_By_Machine",    index=False)
    vt_plan.to_excel(writer,           sheet_name="VT_Plan",               index=False)
    vt_indent_status.to_excel(writer,  sheet_name="VT_Daily_Indent_Status",index=False)
    vt_mach.to_excel(writer,           sheet_name="VT_Machine_Util",       index=False)
    vt_not.to_excel(writer,            sheet_name="VT_Not_Planned",        index=False)
    vt_def.to_excel(writer,            sheet_name="VT_Not_Required_Today", index=False)
    vt_skip.to_excel(writer,           sheet_name="VT_Skipped_Parts",      index=False)
    vt_inv.to_excel(writer,            sheet_name="VT_Inventory_Health",   index=False)
    vt_horizon.to_excel(writer,        sheet_name="VT_Indent_Horizon",     index=False)
    audit_df.to_excel(writer,          sheet_name="VT_Part_Audit",         index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine" in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name != "VT_Plan_By_Machine":
        style_sheet(wb[sheet_name], header_hex)

# ── Extra styling for VT_Daily_Indent_Status ─────────────────
# Colour the two YES/NO status columns cell-by-cell.
# Also bold rows where Meets_Daily_Indent = NO ✗ so they stand out.
if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = (headers_is.index("Meets_Daily_Indent")    + 1
                  if "Meets_Daily_Indent"    in headers_is else None)
    covers_col = (headers_is.index("Covers_Daily_With_Inv") + 1
                  if "Covers_Daily_With_Inv" in headers_is else None)
    gap_col    = (headers_is.index("Gap_vs_Daily")          + 1
                  if "Gap_vs_Daily"          in headers_is else None)

    miss_fill = PatternFill("solid", fgColor="FFC7CE")   # red
    meet_fill = PatternFill("solid", fgColor="C6EFCE")   # green
    gap_fill  = PatternFill("solid", fgColor="FFEB9C")   # amber for partial

    for row in ws_is.iter_rows(min_row=2):
        # Colour Meets_Daily_Indent column
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = meet_fill if str(cell.value) == "YES ✓" else miss_fill
        # Colour Covers_Daily_With_Inv column
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = meet_fill if str(cell.value) == "YES ✓" else gap_fill
        # Bold entire row if production alone misses indent
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

tab_colors = {
    "VT_Plan_By_Machine":       "0D6E6E",
    "VT_Plan":                  "1F4E79",
    "VT_Daily_Indent_Status":   "0F4C2A",
    "VT_Machine_Util":          "375623",
    "VT_Not_Planned":           "7B2C2C",
    "VT_Not_Required_Today":    "7F6000",
    "VT_Skipped_Parts":         "5C3D2E",
    "VT_Inventory_Health":      "4A235A",
    "VT_Indent_Horizon":        "154360",
    "VT_Part_Audit":            "1C3557",
}
for name, color in tab_colors.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 18 — SUMMARY PRINT
# =============================================================

print(f"\n{'='*62}")
print(f"  Smart APS V6 Complete  —  VT Only  —  {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"{'='*62}")
print(f"  Total parts in VT sheet    : {len(all_vt_parts_raw)}")
print(f"  ─────────────────────────────────────────────────────")
for status, count in audit_df["Status"].value_counts().items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<42}: {count:>4}")
print(f"  ─────────────────────────────────────────────────────")
print(f"  planned            = {len(vt_plan):>4}  (of parts that entered scheduler)")
print(f"  not_required_today = {len(vt_def):>4}  (inv covers daily indent)")
print(f"  not_planned        = {len(vt_not):>4}  (needed but no machine capacity)")
if not vt_indent_status.empty:
    meets  = (vt_indent_status["Meets_Daily_Indent"] == "YES ✓").sum()
    misses = (vt_indent_status["Meets_Daily_Indent"] == "NO ✗").sum()
    covers = (vt_indent_status["Covers_Daily_With_Inv"] == "YES ✓").sum()
    print(f"\n  Daily Indent Status (VT_Daily_Indent_Status sheet):")
    print(f"    Production meets daily indent  : {meets:>4} parts  ✓")
    print(f"    Production misses daily indent : {misses:>4} parts  ✗  (see sheet for gap)")
    print(f"    Covered when inv is added      : {covers:>4} parts")
print(f"\n  Output  → {output_path}")
print(f"  State   → {MACHINE_STATE_FILE}")
print(f"\n  NOTE: Update SECTION 1 each morning:")
print(f"        PLANNING_DATE = date(2026, 3, 15)")
print(f"        INDENT_MONTH  = date(2026, 3,  1)   ← only change when month rolls over")


  Smart APS V6  —  VT-Only Indent-Based Planning
  Planning date  : 2026-03-19
  Indent month   : March 2026
  Total days     : 31
  Sundays        : 5
  Working days   : 26  (used as divisor)
  Daily target   : monthly_indent / 26
  Today target   : max(0, daily_indent - inventory)
  Skip rule 1    : daily_indent ≤ 150 qty  (absolute, no exceptions)
  Skip rule 2    : monthly_indent / rate  ≤ 4.0h  (absolute, no exceptions)
  Zero-inv rule  : NONE — skip rules apply even when inventory = 0
  Over-prod cap  : 5 × daily_indent  (inv + produced never exceeds this)
  Changeover     : none on first run (clean slate) — charged from day 2 onwards

Loading data...
  VT sheet columns found:
    Part       → 'Part'
    Cycle time → 'Cycle time'
    Cavity     → 'Cavity'
    Inventory  → 'Inventory'
    Indent     → 'Indent'

  VT parts in sheet            : 142
  Parts with valid rate        : 121
  Parts with zero/missing cycle time : 21
    S41222-022                      cycle_time=nan  (ca